# Validação dos tempos de espera e atendimento

Este notebook compara os tempos informados na `tb_chamados` com os calculados pelos horários tratados na `tb_chamados_hora`, ambas da Silver.

A ideia é medir as diferenças e deixar a conferência registrada. Os resultados correspondem às tabelas disponíveis no momento da execução.


In [0]:
from pyspark.sql import functions as F

catalog = "vcredit"
silver_schema = f"{catalog}.silver"

print(f"Fontes: {silver_schema}.tb_chamados e {silver_schema}.tb_chamados_hora")
display(spark.sql("SELECT current_timestamp() AS data_hora_execucao, current_timezone() AS fuso"))


Fontes: vcredit.silver.tb_chamados e vcredit.silver.tb_chamados_hora


data_hora_execucao,fuso
2026-10-28T22:05:08.510Z,Etc/UTC


## 1 - Leitura das fontes

Os campos informados são os que a Gold usava antes da alteração. Os horários vêm da Silver, depois dos tratamentos de data. Mantemos a abertura original e a marcação de correção do ano para rastrear esse tratamento.

A comparação usa valores tratados da Silver. Ela não substitui uma investigação dos valores brutos na Bronze.


In [0]:
df_chamados = (
    spark.table(f"{silver_schema}.tb_chamados")
    .select(
        F.col("id_chamado").cast("string").alias("id_chamado"),
        "canal",
        F.col("tempo_espera").cast("double").alias("espera_informada_seg"),
        F.col("tempo_atendimento").cast("double").alias("atendimento_informado_seg")
    )
    .distinct()
)

df_horarios = (
    spark.table(f"{silver_schema}.tb_chamados_hora")
    .select(
        F.col("id_chamado").cast("string").alias("id_chamado"),
        "hora_abertura_original",
        "hora_abertura_chamado",
        "hora_inicio_atendimento",
        "hora_finalizacao_atendimento",
        "corr_ano_abertura"
    )
    .distinct()
)


## 2 - Conferência dos IDs

Precisamos de uma linha por chamado em cada fonte. Cópias idênticas da seleção foram removidas na leitura. Se ainda houver mais de uma linha por ID, precisamos investigar antes de seguir.


In [0]:
for nome, df in [("chamados", df_chamados), ("horarios", df_horarios)]:
    nulos = df.filter(
        F.col("id_chamado").isNull() | (F.trim(F.col("id_chamado")) == "")
    ).count()
    duplicados = df.groupBy("id_chamado").count().filter(F.col("count") > 1).count()
    print(f"{nome}: {df.count()} registros, {nulos} IDs vazios e {duplicados} IDs repetidos")
    assert nulos == 0, f"Existem IDs vazios em {nome}. Corrija a origem antes de continuar."
    assert duplicados == 0, f"Existem IDs repetidos em {nome}. Investigue antes de continuar."

sem_horario = df_chamados.join(
    df_horarios.select("id_chamado"), "id_chamado", "left_anti"
).count()
sem_chamado = df_horarios.join(
    df_chamados.select("id_chamado"), "id_chamado", "left_anti"
).count()
print(f"Chamados sem registro de horários: {sem_horario}")
print(f"Registros de horários sem chamado: {sem_chamado}")


chamados: 131838 registros, 0 IDs vazios e 0 IDs repetidos
horarios: 131838 registros, 0 IDs vazios e 0 IDs repetidos
Chamados sem registro de horários: 0
Registros de horários sem chamado: 0


## 3 - Cálculo e classificação

Espera é o início do atendimento menos a abertura. Atendimento é a finalização menos o início. Todos os valores estão em segundos. A diferença é **calculado menos informado**. Um valor positivo indica que o intervalo calculado é maior.

As classificações são: **Igual**, **Divergente**, **Sem informação para comparar** e **Intervalo inválido**. Tempo negativo é inválido. Zero é válido. A comparação é exata em segundos, sem tolerância. Ausência de informação não conta como divergência.


In [0]:
df_comparacao = (
    df_chamados
    .join(df_horarios, "id_chamado", "left")
    .withColumn("mes", F.date_format("hora_abertura_chamado", "yyyy-MM"))
    .withColumn(
        "espera_calculada_seg",
        F.col("hora_inicio_atendimento").cast("long")
        - F.col("hora_abertura_chamado").cast("long")
    )
    .withColumn(
        "atendimento_calculado_seg",
        F.col("hora_finalizacao_atendimento").cast("long")
        - F.col("hora_inicio_atendimento").cast("long")
    )
)

campos = [
    ("Espera", "espera_informada_seg", "espera_calculada_seg"),
    ("Atendimento", "atendimento_informado_seg", "atendimento_calculado_seg")
]

partes = []
for indicador, informado, calculado in campos:
    partes.append(
        df_comparacao.select(
            "id_chamado", "canal", "mes",
            F.lit(indicador).alias("indicador"),
            F.col(informado).alias("informado_seg"),
            F.col(calculado).cast("double").alias("calculado_seg")
        )
    )

df_validacao = (
    partes[0].unionByName(partes[1])
    .withColumn("diferenca_seg", F.col("calculado_seg") - F.col("informado_seg"))
    .withColumn(
        "status",
        F.when(
            (F.col("informado_seg") < 0) | (F.col("calculado_seg") < 0),
            "Intervalo inválido"
        )
        .when(
            F.col("informado_seg").isNull() | F.col("calculado_seg").isNull(),
            "Sem informação para comparar"
        )
        .when(F.col("informado_seg") == F.col("calculado_seg"), "Igual")
        .otherwise("Divergente")
    )
)

total_chamados = df_comparacao.count()
assert total_chamados > 0, "A fonte de chamados está vazia."
assert total_chamados == df_chamados.count(), "O cruzamento alterou o número de chamados."
print(f"Total de chamados avaliados: {total_chamados}")


Total de chamados avaliados: 131838


## 4 - Resumo da qualidade

O primeiro quadro mostra a quantidade de cada classificação e sua participação no total de chamados. Espera e atendimento são avaliados separadamente. Não some suas divergências para contar chamados únicos, pois um chamado pode divergir nos dois tempos.


In [0]:
df_resumo = (
    df_validacao.groupBy("indicador", "status").count()
    .withColumnRenamed("count", "quantidade")
    .withColumn("percentual_total", F.round(F.col("quantidade") / total_chamados * 100, 2))
)
display(df_resumo.orderBy("indicador", "status"))


indicador,status,quantidade,percentual_total
Atendimento,Divergente,27986,21.23
Atendimento,Igual,99388,75.39
Atendimento,Intervalo inválido,1,0.0
Atendimento,Sem informação para comparar,4463,3.39
Espera,Divergente,6107,4.63
Espera,Igual,46757,35.47
Espera,Sem informação para comparar,78974,59.9


## 5 - Percentual de divergência entre registros comparáveis

Aqui entram apenas registros classificados como iguais ou divergentes. Os casos sem informação e os negativos ficam fora do denominador. Esta é a taxa para citar no relatório, sempre junto da quantidade de registros comparáveis.


In [0]:
df_taxa_divergencia = (
    df_validacao.groupBy("indicador")
    .agg(
        F.sum(F.when(F.col("status").isin("Igual", "Divergente"), 1).otherwise(0))
         .alias("registros_comparaveis"),
        F.sum(F.when(F.col("status") == "Divergente", 1).otherwise(0))
         .alias("registros_divergentes")
    )
    .withColumn(
        "percentual_divergente",
        F.when(
            F.col("registros_comparaveis") > 0,
            F.round(F.col("registros_divergentes") / F.col("registros_comparaveis") * 100, 2)
        )
    )
)
display(df_taxa_divergencia.orderBy("indicador"))


indicador,registros_comparaveis,registros_divergentes,percentual_divergente
Atendimento,127374,27986,21.97
Espera,52864,6107,11.55


## 6 - Tamanho das diferenças

Este quadro usa somente registros em que os dois tempos são não negativos e estão disponíveis. As duas médias usam exatamente os mesmos chamados. A diferença absoluta mostra o tamanho do desvio sem compensar valores positivos e negativos.


In [0]:
display(
    df_validacao.filter(F.col("status").isin("Igual", "Divergente"))
    .groupBy("indicador")
    .agg(
        F.count("*").alias("registros_comparaveis"),
        F.round(F.avg("informado_seg"), 2).alias("media_informada_seg"),
        F.round(F.avg("calculado_seg"), 2).alias("media_calculada_seg"),
        F.round(F.avg(F.abs(F.col("diferenca_seg"))), 2).alias("diferenca_absoluta_media_seg"),
        F.max(F.abs(F.col("diferenca_seg"))).alias("maior_diferenca_absoluta_seg")
    )
    .orderBy("indicador")
)


indicador,registros_comparaveis,media_informada_seg,media_calculada_seg,diferenca_absoluta_media_seg,maior_diferenca_absoluta_seg
Atendimento,127374,216.6,662.29,445.69,8991.0
Espera,52864,344.77,519.5,174.73,6226.0


## 7 - Diferenças por canal e mês

Os percentuais usam os registros comparáveis de cada grupo. Os quadros ajudam a localizar o problema sem confundir um canal com mais volume com um canal proporcionalmente mais afetado.


In [0]:
def resumo_por(coluna):
    return (
        df_validacao.groupBy("indicador", coluna)
        .agg(
            F.count("*").alias("total_chamados"),
            F.sum(F.when(F.col("status").isin("Igual", "Divergente"), 1).otherwise(0))
             .alias("comparaveis"),
            F.sum(F.when(F.col("status") == "Divergente", 1).otherwise(0))
             .alias("divergentes")
        )
        .withColumn(
            "percentual_divergente",
            F.when(F.col("comparaveis") > 0, F.round(F.col("divergentes") / F.col("comparaveis") * 100, 2))
        )
        .orderBy("indicador", coluna)
    )

display(resumo_por("canal"))
display(resumo_por("mes"))


indicador,canal,total_chamados,comparaveis,divergentes,percentual_divergente
Atendimento,Atendimento Especializado,17091,15321,13177,86.01
Atendimento,Atendimento Inicial,39697,37653,14805,39.32
Atendimento,Chatbot,35517,35223,0,0.0
Atendimento,URA,39533,39177,4,0.01
Espera,Atendimento Especializado,17091,15290,4423,28.93
Espera,Atendimento Inicial,39697,37574,1684,4.48
Espera,Chatbot,35517,0,0,null
Espera,URA,39533,0,0,null


indicador,mes,total_chamados,comparaveis,divergentes,percentual_divergente
Atendimento,2025-01,4368,4273,837,19.59
Atendimento,2025-02,4689,4607,948,20.58
Atendimento,2025-03,4925,4814,983,20.42
Atendimento,2025-04,5220,5092,1078,21.17
Atendimento,2025-05,5508,5367,1134,21.13
Atendimento,2025-06,5824,5670,1240,21.87
Atendimento,2025-07,6069,5895,1263,21.42
Atendimento,2025-08,6323,6130,1339,21.84
Atendimento,2025-09,6557,6366,1329,20.88
Atendimento,2025-10,6942,6707,1449,21.6


## 8 - Exemplos para conferir

Os horários ao lado permitem conferir a conta. A abertura original e a marcação de correção do ano ajudam a identificar efeitos do tratamento de datas.


In [0]:
for indicador in ["Espera", "Atendimento"]:
    print(f"Exemplos de divergência: {indicador}")
    display(
        df_validacao
        .filter((F.col("indicador") == indicador) & (F.col("status") == "Divergente"))
        .join(
            df_horarios.select(
                "id_chamado", "hora_abertura_original", "hora_abertura_chamado",
                "hora_inicio_atendimento", "hora_finalizacao_atendimento", "corr_ano_abertura"
            ),
            "id_chamado", "left"
        )
        .orderBy(F.abs(F.col("diferenca_seg")).desc(), "id_chamado")
        .limit(30)
    )

print("Registros com tempo negativo")
display(df_validacao.filter(F.col("status") == "Intervalo inválido").orderBy("indicador", "id_chamado"))


Exemplos de divergência: Espera


id_chamado,canal,mes,indicador,informado_seg,calculado_seg,diferenca_seg,status,hora_abertura_original,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,corr_ano_abertura
87852,Atendimento Especializado,2026-03,Espera,6.0,6232.0,6226.0,Divergente,2026-03-05T10:49:30.000Z,2026-03-05T10:49:30.000Z,2026-03-05T12:33:22.000Z,2026-03-05T12:50:14.000Z,false
123649,Atendimento Especializado,2026-07,Espera,5.0,5959.0,5954.0,Divergente,2026-07-27T22:00:36.000Z,2026-07-27T22:00:36.000Z,2026-07-27T23:39:55.000Z,2026-07-27T23:54:50.000Z,false
76323,Atendimento Especializado,2026-01,Espera,5.0,5444.0,5439.0,Divergente,2026-01-20T10:48:06.000Z,2026-01-20T10:48:06.000Z,2026-01-20T12:18:50.000Z,2026-01-20T12:37:34.000Z,false
124079,Atendimento Especializado,2026-07,Espera,5.0,5304.0,5299.0,Divergente,2026-07-07T19:17:35.000Z,2026-07-07T19:17:35.000Z,2026-07-07T20:45:59.000Z,2026-07-07T21:29:18.000Z,false
124332,Atendimento Especializado,2026-07,Espera,5.0,5016.0,5011.0,Divergente,2026-07-31T12:07:29.000Z,2026-07-31T12:07:29.000Z,2026-07-31T13:31:05.000Z,2026-07-31T14:08:35.000Z,false
89381,Atendimento Especializado,2026-03,Espera,4.0,4984.0,4980.0,Divergente,2026-03-23T10:20:57.000Z,2026-03-23T10:20:57.000Z,2026-03-23T11:44:01.000Z,2026-03-23T12:08:55.000Z,false
128595,Atendimento Especializado,2026-07,Espera,4.0,4598.0,4594.0,Divergente,2026-07-09T18:42:25.000Z,2026-07-09T18:42:25.000Z,2026-07-09T19:59:03.000Z,2026-07-09T20:20:09.000Z,false
86125,Atendimento Especializado,2026-02,Espera,4.0,4589.0,4585.0,Divergente,2026-02-06T06:00:06.000Z,2026-02-06T06:00:06.000Z,2026-02-06T07:16:35.000Z,2026-02-06T07:49:22.000Z,false
105287,Atendimento Especializado,2026-05,Espera,4.0,4550.0,4546.0,Divergente,2026-05-25T15:31:43.000Z,2026-05-25T15:31:43.000Z,2026-05-25T16:47:33.000Z,2026-05-25T17:20:44.000Z,false
124807,Atendimento Especializado,2026-07,Espera,4.0,4500.0,4496.0,Divergente,2026-07-30T08:44:54.000Z,2026-07-30T08:44:54.000Z,2026-07-30T09:59:54.000Z,2026-07-30T10:27:24.000Z,false


Exemplos de divergência: Atendimento


id_chamado,canal,mes,indicador,informado_seg,calculado_seg,diferenca_seg,status,hora_abertura_original,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,corr_ano_abertura
102699,Atendimento Especializado,2026-04,Atendimento,9.0,9000.0,8991.0,Divergente,2026-04-24T08:51:25.000Z,2026-04-24T08:51:25.000Z,2026-04-24T08:59:00.000Z,2026-04-24T11:29:00.000Z,false
104707,Atendimento Especializado,2026-05,Atendimento,9.0,9000.0,8991.0,Divergente,2026-05-08T19:05:51.000Z,2026-05-08T19:05:51.000Z,2026-05-08T19:39:07.000Z,2026-05-08T22:09:07.000Z,false
105280,Atendimento Especializado,2026-05,Atendimento,9.0,9000.0,8991.0,Divergente,2026-05-05T09:25:01.000Z,2026-05-05T09:25:01.000Z,2026-05-05T09:38:37.000Z,2026-05-05T12:08:37.000Z,false
11180,Atendimento Especializado,2025-03,Atendimento,9.0,9000.0,8991.0,Divergente,2025-03-31T14:51:16.000Z,2025-03-31T14:51:16.000Z,2025-03-31T15:00:27.000Z,2025-03-31T17:30:27.000Z,false
112253,Atendimento Especializado,2026-05,Atendimento,9.0,9000.0,8991.0,Divergente,2026-05-27T15:58:25.000Z,2026-05-27T15:58:25.000Z,2026-05-27T16:46:13.000Z,2026-05-27T19:16:13.000Z,false
113253,Atendimento Especializado,2026-06,Atendimento,9.0,9000.0,8991.0,Divergente,2026-06-22T10:32:02.000Z,2026-06-22T10:32:02.000Z,2026-06-22T10:37:09.000Z,2026-06-22T13:07:09.000Z,false
122228,Atendimento Especializado,2026-06,Atendimento,9.0,9000.0,8991.0,Divergente,2026-06-02T03:58:47.000Z,2026-06-02T03:58:47.000Z,2026-06-02T04:18:23.000Z,2026-06-02T06:48:23.000Z,false
126506,Atendimento Especializado,2026-07,Atendimento,9.0,9000.0,8991.0,Divergente,2026-07-23T13:13:47.000Z,2026-07-23T13:13:47.000Z,2026-07-23T14:00:59.000Z,2026-07-23T16:30:59.000Z,false
127842,Atendimento Especializado,2026-07,Atendimento,9.0,9000.0,8991.0,Divergente,2026-07-02T16:23:43.000Z,2026-07-02T16:23:43.000Z,2026-07-02T16:33:55.000Z,2026-07-02T19:03:55.000Z,false
14469,Atendimento Especializado,2025-04,Atendimento,9.0,9000.0,8991.0,Divergente,2025-04-21T16:11:39.000Z,2025-04-21T16:11:39.000Z,2025-04-21T16:17:40.000Z,2025-04-21T18:47:40.000Z,false


Registros com tempo negativo


id_chamado,canal,mes,indicador,informado_seg,calculado_seg,diferenca_seg,status
107034,Atendimento Inicial,2026-05,Atendimento,1.0,-84503.0,-84504.0,Intervalo inválido


## 9 - Resultado da execução

O texto abaixo resume a comparação realizada nesta execução. Os números podem diferir de outras conferências quando as fontes forem atualizadas ou os critérios forem diferentes.


In [0]:
for linha in df_taxa_divergencia.orderBy("indicador").collect():
    taxa = linha["percentual_divergente"]
    percentual = f"{taxa:.2f}%" if taxa is not None else "não calculável"
    print(
        f"{linha['indicador']}: {linha['registros_divergentes']} divergências "
        f"em {linha['registros_comparaveis']} registros comparáveis ({percentual})."
    )
print(f"Chamados sem horários correspondentes: {sem_horario}.")
print(f"Registros de horários sem chamado correspondente: {sem_chamado}.")
print("A comparação terminou. Nenhuma tabela foi alterada.")


Atendimento: 27986 divergências em 127374 registros comparáveis (21.97%).
Espera: 6107 divergências em 52864 registros comparáveis (11.55%).
Chamados sem horários correspondentes: 0.
Registros de horários sem chamado correspondente: 0.
A comparação terminou. Nenhuma tabela foi alterada.


## 10 - Conferência dos cálculos da Silver

Agora verificamos se espera_seg e atendimento_seg correspondem aos horários tratados. A comparação considera dois nulos como equivalentes. As flags devem ser verdadeiras apenas para intervalos não negativos e disponíveis.

As contas usam os mesmos horários tratados da tabela. Este teste confirma a implementação, mas não valida sozinho se os horários representam os eventos corretos do negócio.


In [0]:
df_silver_check = (
    spark.table(f"{silver_schema}.tb_chamados_hora")
    .withColumn("espera_recalculada", F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("atendimento_recalculado", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
)
problemas_silver = df_silver_check.filter(
    (~F.col("espera_seg").eqNullSafe(F.col("espera_recalculada")))
    | (~F.col("atendimento_seg").eqNullSafe(F.col("atendimento_recalculado")))
    | (~F.col("espera_valida").eqNullSafe(F.coalesce(F.col("espera_recalculada") >= 0, F.lit(False))))
    | (~F.col("atendimento_valido").eqNullSafe(F.coalesce(F.col("atendimento_recalculado") >= 0, F.lit(False))))
)
qtd_problemas_silver = problemas_silver.count()
print(f"Registros com cálculo ou flag diferente da regra: {qtd_problemas_silver}")
display(problemas_silver.select("id_chamado", "espera_seg", "espera_recalculada", "espera_valida", "atendimento_seg", "atendimento_recalculado", "atendimento_valido").limit(30))
assert qtd_problemas_silver == 0, "A Silver não corresponde à regra. Investigue os exemplos antes de validar a Gold."


Registros com cálculo ou flag diferente da regra: 0


id_chamado,espera_seg,espera_recalculada,espera_valida,atendimento_seg,atendimento_recalculado,atendimento_valido


## 11 - Conferência da Gold gravada

Esta etapa lê a fato já salva. Não usa o DataFrame temporário de outro notebook. O valor esperado é o intervalo da Silver quando a flag é válida; nos demais casos, é nulo.

O teste usa todos os chamados da fonte e da fato. Assim, também detecta chamados que sumiram ou foram acrescentados. Dois nulos são equivalentes, mas nulo e zero são diferentes.


In [0]:
df_gold_check = (
    spark.table(f"{catalog}.gold.fact_chamados")
    .select(
        F.col("id_chamado").cast("string").alias("id_chamado"),
        F.col("tempo_espera_seg").cast("decimal(18,2)").alias("espera_gold"),
        F.col("tempo_atendimento_seg").cast("decimal(18,2)").alias("atendimento_gold")
    )
)
qtd_ids_invalidos_gold = df_gold_check.filter(F.col("id_chamado").isNull() | (F.trim(F.col("id_chamado")) == "")).count()
qtd_repetidos_gold = df_gold_check.groupBy("id_chamado").count().filter(F.col("count") > 1).count()
assert qtd_ids_invalidos_gold == 0, "Existem IDs vazios na fato."
assert qtd_repetidos_gold == 0, "Existem IDs repetidos na fato. Confira os joins."

df_esperado = (
    df_chamados.select("id_chamado")
    .join(
        df_silver_check.select(
            F.col("id_chamado").cast("string").alias("id_chamado"),
            F.when(F.col("espera_valida"), F.col("espera_seg")).cast("decimal(18,2)").alias("espera_esperada"),
            F.when(F.col("atendimento_valido"), F.col("atendimento_seg")).cast("decimal(18,2)").alias("atendimento_esperado")
        ),
        "id_chamado", "left"
    )
    .withColumn("existe_na_fonte", F.lit(True))
)

df_conferencia_gold = (
    df_esperado.join(
        df_gold_check.withColumn("existe_na_gold", F.lit(True)),
        "id_chamado", "full"
    )
    .withColumn("problema_espera", ~F.col("espera_esperada").eqNullSafe(F.col("espera_gold")))
    .withColumn("problema_atendimento", ~F.col("atendimento_esperado").eqNullSafe(F.col("atendimento_gold")))
)

df_problemas_gold = df_conferencia_gold.filter(
    F.col("existe_na_fonte").isNull()
    | F.col("existe_na_gold").isNull()
    | F.col("problema_espera")
    | F.col("problema_atendimento")
)
qtd_problemas_gold = df_problemas_gold.count()
print(f"Chamados na fonte: {df_chamados.count()}")
print(f"Chamados na Gold: {df_gold_check.count()}")
print(f"Chamados ausentes, extras ou com tempos fora da regra: {qtd_problemas_gold}")
display(df_problemas_gold.orderBy("id_chamado").limit(50))
assert qtd_problemas_gold == 0, "A Gold gravada não corresponde à regra. Confira se a célula corrigida foi executada e gravou a fato."
print("Validação concluída: a Gold gravada corresponde aos tempos válidos da Silver.")


Chamados na fonte: 131838
Chamados na Gold: 131838
Chamados ausentes, extras ou com tempos fora da regra: 0


id_chamado,espera_esperada,atendimento_esperado,existe_na_fonte,espera_gold,atendimento_gold,existe_na_gold,problema_espera,problema_atendimento


Validação concluída: a Gold gravada corresponde aos tempos válidos da Silver.


## 12 - Conclusão

Identificamos diferenças entre os tempos informados na tb_chamados e os intervalos calculados pelos horários tratados na Silver. Os valores originais permanecem preservados na Bronze. A regra adotada usa a diferença entre abertura e início para espera, e entre início e finalização para atendimento, em segundos. A Gold recebe apenas os intervalos válidos. Valores negativos ou sem informação ficam nulos; zero permanece válido.

Os quadros anteriores registram as quantidades, os percentuais e os exemplos desta execução. Alguns exemplos são compatíveis com perda de informação numérica na origem. A causa permanece em investigação até confirmação. Este notebook não altera tabelas.

Antes de encerrar a documentação, preencha a conferência da equipe na última célula.
